# Introduction to Internet

Let's begin by learning how the internet works. The Internet is the backbone of the Web - it is the technical infrastructure that makes the Web possible. A simple, working model of the Internet is that it is a large network of computers which communicate together. Computers on the internet adhere to a client-server model.

![Client-server model](figs/simple-client-server.png){width=40%}

In this topic, we are going to use Python to perform both sides of the operation:

::: {layout-ncol=2}
### Client side

- Use the `requests` package to send requests and receive responses from servers

### Server side

- Use `Flask` to run a server to handle requests
  
:::

Each computer on the internet is assigned a unique IP address. IP addresses can be released and re-used. There are two versions of the Internet Protocol: version 4 and version 6. Although version 6 was supposed to supplant the earlier one some time ago, IPv4 is still widely used. An IP address (v4) consists of four integers separated by periods. Each integer is between 0 and 255.

::: {.callout-note}

How to check the IP address of your computer?

:::

There are a few special IP addresses you should know about:

* `127.0.0.1` corresponds to your local host. If you send a request to this address, it never actually leaves your computer.
* `localhost` is a pseduonym for this address.
* Some addresses are only used on internal networks (behind a router), e.g. 192.168.x.x and 10.x.x.x

A URL is an address for a resource on the internet. Here are some examples:

* `http://localhost:8888/lab/tree/src/api_workshop`
* `https://www.google.com`
* `https://www.theguardian.com/us-news/us-politics `

# HTTP Requests and Responses 

## HTTP Requests

HTTP (hypertext transfer protocol) is the main protocol on the internet. It is used for transferring websites and images. Communication in HTTP revolves around requests and responses. The client sends a request, and the server sends a response. A valid request requires:

1. URL (Uniform resource locator)
    * A URL is a unique address for an object, known as a *resource*, on the server. 
    * When we browse web-pages, each of them has a unique address. When we work with an API, an object could be a *product*, or a *customer*.
2. Method
   * The four most common methods are:
       * GET: to ask the server to retrieve a resource
       * POST: to ask the server to create a new resource
       * PUT: to ask the server to edit/update an existing resource
       * DELETE: to ask the server to delete a resource.
3. List of headers
    * Headers provide meta-information about a request. Think of them as key-value pairs.
    * Suppose you direct your mobile phone browser to a web-page. That browser will send a GET request to the web-server to obtain the page. When it does so, it includes information in the "User-Agent" header that it is running on a mobile phone. The web-server can react to this and send a version of the page that is optimised for a mobile phone browser.
4. Body
    * Contains the data that the client wants to send to the server.
    * Here's what it looks like:

![HTTP request components](figs/http_request_components.jpeg){width=40%}

## HTTP Responses

HTTP responses are very similar to requests, but instead of a method and a url, the response contains a status code. Responses are made up of

1. Status code
    * These are 3-digit numbers that have a unique meaning. Examples are 
        * 404: resource not found
        * 200: successful request.
        * 201: successfully created a resource on the server.
    * Here is a full list of [response codes](https://developer.mozilla.org/en-US/docs/Web/HTTP/Status).
2. Headers
3. Body

Here's an image of a response:

![HTTP Response](figs/http_response_content.jpeg){width=40%}


### Data Formats

APIs typically use either JSON or XML to format the data before sending it. The data format is specified in the *Content-Type* header field.

For instance, when the client sends a request with JSON data, it will include the value "application/json" in the Content-Type header field. The client can also specify the content that it expects/can work with. This is done in the *Accept* header field.

When the server responds, it will also include a specification of it's data format in the response header.

![Content type](figs/http_request_content.jpeg){width=40%}

## Making Requests

Here are the examples from the Canvas videos.

In [2]:
import requests

## Example 1

In [2]:
# Request 1: Random Joke
response_joke = requests.get('https://official-joke-api.appspot.com/random_joke')
if response_joke.status_code == 200:
    joke = response_joke.json()
    print("Joke:")
    print(f"""
    Q: {joke['setup']}
    A: {joke['punchline']}
    """)
else:
    print(f"Joke API request failed with status code: {response_joke.status_code}")

Joke:

    Q: Why was the designer always cold?
    A: Because they always used too much ice-olation.
    


## Example 2

In [3]:
# Request 2: 24-hour weather forecast from data.gov.sg
response_forecast = requests.get('https://api-open.data.gov.sg/v2/real-time/api/twenty-four-hr-forecast')
if response_forecast.status_code == 200:
    region = 'central'
    print(f"Forecast for {region} region:")
    for fc in response_forecast.json()['data']['records'][0]['periods']:
        print(f"* {fc['timePeriod']['text']}:\t {fc['regions'][region]['text']}")    
else:
    print(f"Forecast retrieval failed: {response_forecast.status_code}")

Forecast for central region:
* 6 pm 08 Aug to 6 am 09 Aug:	 Partly Cloudy (Night)
* 6 am to Midday 09 Aug:	 Partly Cloudy (Day)
* Midday to 6 pm 09 Aug:	 Partly Cloudy (Day)


# Flask Applications

## Basics

The file `demo.py` contains the demonstration app from the third Canvas video.

In [4]:
#| eval: false

from flask import Flask, jsonify                                 
 
app = Flask(__name__)                                            #<1>
  
@app.route('/api')                                               #<2>
def demo_fn():                                                   #<3>  
    return jsonify({'Hello': 'World!'})

@app.route('/api/<int:x>')                                       #<4>
def demo_fn2(x): 
    return jsonify({'Hello': 'World!', 'Number': f'{x}'})

1.  Instantiates the Flask app
2.  Defines the first endpoint, a GET method.
3.  The first view function, that will be triggered when a request is sent to the endpoint.
4.  A URL that can be parsed to retrieve an integer, which will be passed to the second view function.

To run the Flask app, run the following command (within Jupyter). You can also run it from a terminal (without the '!', but make sure your environment is activated).

In [5]:
#!flask -A demo.py run --reload 

::: {.callout-note}

How to debug a Flask application?

:::

In [3]:
u1 = "http://localhost:5000/api"
r1 = requests.get(u1)

In [5]:
r1.json()

{'Hello': 'World!'}

## Decorators

The line above the view function is a decorator. In Python, decorators are used to modify the behaviour of functions. The decorator is usually the name of a function that takes another function as an argument, and returns a function. Here is an example of how decorators work:

## Example 3

In [6]:
import functools

def uppercase_decorator(func):
    @functools.wraps(func)
    def wrapper():
        return func().upper()
    return wrapper

@uppercase_decorator
def say_hi():
    "This will say hi"
    return 'hello there'

say_hi()

'HELLO THERE'

When we call a decorator, we are doing just a little bit more than this:

In [7]:
def uppercase_decorator(func):
    def wrapper():
        return func().upper()
    return wrapper
    
def say_hi():
    "This will say hi"
    return 'hello there'
    
say_hi = uppercase_decorator(say_hi)
say_hi()

'HELLO THERE'

Flask decorators do not really modify the behaviour of the function greatly. Their more important task is to register the URL in a mapping table, so that the server can trigger the appropriate function.

## session and g Objects

The `session` and `g` objects in Flask are special objects. The `g` object is the correct place to store resources that can be used across functions. It looks like a global object, but it is thread safe and request-specific!

In [8]:
#| eval: false
from flask import Flask, jsonify, g
from datetime import datetime
 
app = Flask(__name__)                                            

@app.before_request                                              #<1> 
def before_request():
    g.request_start_time = datetime.now()
    
@app.route('/api')                                               
def demo_fn():                                                     
    return jsonify({'Hello': 'World!',
                    'time' : g.request_start_time.isoformat(sep=' ')})

1.  A function that runs *before* any view function.

On the other hand, the `session` object persists across requests from a single user. This can be used to sustain an authenticated session for a user.

In [9]:
#| eval: false
from flask import Flask, jsonify, session
 
app = Flask(__name__)                                            
app.secret_key = b'_5#y2L"F4Q8z\n\xec]/'
    
@app.route('/api')                                               
def demo_fn():
    if 'calls' not in session:
        session['calls'] = 1
    else:
        session['calls'] += 1
    return f"{session['calls']} made so far..."

@app.route('/api/<int:x>')                                       
def demo_fn2(x): 
    if 'calls' not in session:
        session['calls'] = 1
    else:
        session['calls'] += 1
    return f"{session['calls']} made so far..."

@app.route('/clear')
def logout():
    session.pop('calls', None)
    return "<H2> Session cleared!! </H2>"

## Dockerising a Flask Application

In this course, we always need to practice deploying our applications. Here is a simple Dockerfile to deploy the simple app we have been working with.

::: {.panel-tabset}

### Dockerfile

In [ ]:
#| eval: false
FROM python:3.11-slim

WORKDIR /app

COPY demo.py demo.py

RUN pip3 install --upgrade pip
RUN pip3 install flask flasgger

ENTRYPOINT [ "python", "-u", "-m", "flask", "-A", "demo", "run", "--host=0.0.0.0"]

### Docker compose

In [ ]:
#| eval: false
services:
  flask-app:
    build: .
    ports:
      - "5000:5000"

:::

## Example 4: OAuth

OAuth is a protocol that is very commonly used for authenticating user. Every time you visit a site that says "Sign in with Google" or "Sign in with Github", you are in fact using OAuth. 

![OAuth Example](figs/oauth.png){width="50%"}

Briefly, OAuth is a protocol that allows your application to pass on the responsibility of authenticating a user to someone else. The user may still have an account with your app, but you do not have to store long-term credentials, or passwords any more. For your projects, do consider this approach in your applications instead of implementing a password login. The latter requires strong expertise in network security.

Here is a brief overview of the steps involved in OAuth. There are three parties involved in OAuth: the user, your application (consumer) and the OAuth provider.

1.  Client visits the consumer application and indicates desire to authenticate.
2.  Consumer redirects the client to the provider, listing the permissions (*scope*) the consumer app needs.
3.  Client grants permission to access his/her information from the provider.
4.  Permission is forwarded to the consumer, along with a secret.
5.  Consumer uses secret to ask for a token from the provider. The token is a short term credential, with limited scope on the user account with the provider.
6.  Consumer stores the token and uses it in requests made to the provider.

![OAuth Example](figs/oauth2_flow.png){width="50%"}


As an example, consider that you are writing a simple application that will retrieve the list of github repositories for which you are a collaborator. In that case, your application would request for a token from github, on behalf of the user, that has read access to your profile and list of repositories on github. Take note that there are applications that simply use the provider to authenticate, retrieving a token with minimal access, e.g. username and email.

```
oauth2/
├── cert.csr
├── cert.pem
├── github.py
└── key.pem
```

Take the opportunity to get comfortable with some common Flask methods. The application uses some of them:

*   `render_template`
*   `url_for`
*   `redirect`
*   Blueprints
*   Storing secrets
*   Enforcing https instead of http



# Group Activity: What is Data Science?

The `ds-19` folder contains a back-end developed by one of the project groups in DSA3101 a few semesters ago. They had been tasked with creating a portal that would help university applicants understand the difference between data science departments of the various universities in Singapore. They had named their solution "Datacompass". The files here are a modified version of their code. In particular, the front-end has been removed.

```
ds-19/
├── backend
│   ├── ntu.csv
│   ├── nus-dsa.csv
│   ├── nus-dse.csv
│   └── smu.csv
├── datacompass.py
└── utils.py

```

The flask application you will be working with is in `datacompass.py`. There are three endpoints there:

| View function         | Methods | Rule                               |
|-----------------------|---------|-------------------------------------|
| check_prereq          | POST    | /prereq/<string:mod_code>           |
| compute_sim_concepts  | GET     | /similarity                         |
| get_graph             | GET     | /getgraph/<string:mod_code>         |

The file `utils.py` contains utility functions that are used in the flask application.

::: {.callout-warning}

Full disclosure: the functions in `utils.py` were largely written by chatGPT. I have tested them, but *caveat emptor*!

:::

## Learning outcomes

* To practice with Flask:
  * Running an application
  * Creating a new endpoint
  * Debugging an application
  * Testing an application's endpoints
* To practice with Docker:
  * Installing things on Linux
  * Navigating a Linux container
  * Constructing a Dockerfile, and a corresponding docker-compose file.

## Demonstrating endpoints

Here is example code for testing the current endpoints in the application.

### Pre-req endpoint

In [22]:
import requests

url1 = "http://localhost:5000/prereq/ST3131"
r1 = requests.post(url1, json='["ST1131:A", "ST2131:D"]')
if r1.json():
    print("Pre-req pass.")
else:
    print("Pre-req fail.")

Pre-req pass.


### Pre-req tree

In [24]:
url1 = "http://localhost:5000/getgraph/ST3236"
r1 = requests.get(url1)

with open('new_image2.png', 'wb') as f:
    f.write(r1.content)

### Similarity

In [25]:
url1 = "http://localhost:5000/similarity/concepts"
params = {'uni1': 'nus',
          'mod1': 'DSA2101',
          'uni2': 'ntu',
          'mod2': 'SC4024'}
r1 = requests.get(url1, params=params)
print(f"Similarity: {r1.json():.2f}.")

Similarity: 0.33.


In [26]:
url1 = "http://localhost:5000/similarity/titles"
params = {'uni1': 'nus',
          'mod1': 'DSA2101',
          'uni2': 'ntu',
          'mod2': 'SC4024'}
r1 = requests.get(url1, params=params)
print(f"Similarity: {r1.json():.2f}.")

Similarity: 0.53.


::: {.callout-tip}

You have two tasks here:

1. Docker-ise the application. Take the opportunity to study and identify areas of improvement of the application.
2. Add one endpoint, that computes the similarity between module titles, using Levenshtein distance.

Here is a suggested workflow:

1.	Get the application running on your local machine
2.	Start a python/ubuntu/linux container. Get the application running there.
3.	Start constructing a Dockerfile.
4.	Create a docker-compose.yml file.

:::

# References

1. [Flask documentation](https://flask.palletsprojects.com/en/stable/)
2. [Flask-dance documentation](https://flask-dance.readthedocs.io/en/latest/)
3. [Requests documentation](https://requests.readthedocs.io/en/latest/)
4. [Flasgger github site](https://github.com/flasgger/flasgger)
5. Regarding decorators in Python:
   * From the official Python docs: [functools](https://docs.python.org/3/library/functools.html)
   * From datacamp tutorial: [Python decorators](https://www.datacamp.com/tutorial/decorators-python)
6. More about OAuth:
   * [From Microsoft](https://www.microsoft.com/en-us/security/business/security-101/what-is-oauth)
   * [From Flask-Dance](https://flask-dance.readthedocs.io/en/latest/how-oauth-works.html)